# 🦆 Microduck Bipedal Robot Reinforcement Learning (Google Colab & Drive)

本 Notebook 提供在 **Google Colab** 雲端環境上訓練與評估 **Pollen Robotics Microduck 14-DOF 雙足行走機器人** 的完整實驗流程。

### 🌟 核心特色：
1. **Google Drive 永久空間存放**：自動掛載 Google Drive，建立專屬專案資料夾，所有訓練權重、日誌與錄製影片永久保存，不因 Colab 斷線而丟失！
2. **8 影分身平行向量化訓練 (8 Clones PPO)**：在 Colab GPU/CPU 上平行模擬 8 隻小鴨，快速收斂。
3. **高畫質 HUD 步態影片即時播放**：直接在 Notebook 儲存格中播放帶有遙測儀表板的 MP4 步態影片。

## 步驟 1：掛載 Google Drive 並建立專案永久存放空間

執行此單元格後，請授權 Google Drive 存取。這將在您的雲端硬碟建立 `/MyDrive/microduck_training/` 資料夾。

In [ ]:
import os
from google.colab import drive

# 1. 掛載 Google Drive
print("📂 正在掛載 Google Drive...")
drive.mount('/content/drive')

# 2. 在 Google Drive 中建立專案儲存空間
GDRIVE_DIR = '/content/drive/MyDrive/microduck_training'
os.makedirs(GDRIVE_DIR, exist_ok=True)
os.makedirs(f"{GDRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{GDRIVE_DIR}/logs", exist_ok=True)
os.makedirs(f"{GDRIVE_DIR}/videos", exist_ok=True)

print(f"✅ Google Drive 專案空間已就緒：{GDRIVE_DIR}")
os.chdir(GDRIVE_DIR)
print(f"當前工作目錄：{os.getcwd()}")

## 步驟 2：安裝 Linux 無頭渲染與強化學習依賴套件

Google Colab 為無實體螢幕的 Linux 環境，我們安裝 `libosmesa6-dev` 與最新版 `mujoco`、`gymnasium` 及 `stable-baselines3`。

In [ ]:
# 1. 安裝系統渲染依賴 (OSMesa / EGL / FFmpeg)
!apt-get update -qq && apt-get install -y -qq libosmesa6-dev ffmpeg patchelf

# 2. 安裝 Python 核心強化學習庫
!pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 stable-baselines3>=2.2.0 opencv-python-headless rich imageio[ffmpeg] pillow

# 3. 設定 MuJoCo 無頭渲染環境變數 (GPU 啟用 EGL，CPU 啟用 OSMesa)
import os, torch
if torch.cuda.is_available():
    os.environ["MUJOCO_GL"] = "egl"
    print(f"🚀 偵測到 GPU 加速器：{torch.cuda.get_device_name(0)}，啟用 EGL 渲染硬體加速。")
else:
    os.environ["MUJOCO_GL"] = "osmesa"
    print("⚡ 運行於 CPU 實例，啟用 OSMesa 無頭軟體渲染。")

import mujoco, gymnasium, stable_baselines3
print(f"✅ 套件載入成功！MuJoCo 版本: {mujoco.__version__}, SB3 版本: {stable_baselines3.__version__}")

## 步驟 3：解壓縮或下載 Microduck 模型與環境程式碼

如果您已將 `microduck_colab_package.zip` 上傳至 Google Drive 或 Colab，直接解壓縮；如果尚未上傳，下方腳本會自動同步。

In [ ]:
# 檢查程式碼是否已存在於 Google Drive 中
if not os.path.exists(f"{GDRIVE_DIR}/envs/microduck_env.py"):
    print("📦 正在從 GitHub (Child-pi/microduck, branch: rl-locomotion) 同步強化學習套件...")
    !git clone -b rl-locomotion --depth 1 https://github.com/Child-pi/microduck.git /tmp/microduck_repo
    !cp -r /tmp/microduck_repo/rl_training/* {GDRIVE_DIR}/
    !rm -rf /tmp/microduck_repo
    print("✅ 同步完成！")
elif os.path.exists("/content/microduck_colab_package.zip"):
    !unzip -q -o /content/microduck_colab_package.zip -d {GDRIVE_DIR}

# 加入 Python 模組搜索路徑
import sys
if GDRIVE_DIR not in sys.path:
    sys.path.append(GDRIVE_DIR)

# 驗證檔案結構
!ls -lh {GDRIVE_DIR}


## 步驟 4：測試 Gymnasium 強化學習環境與最佳預訓練模型

測試 14-DOF 機器人步態環境能否正常 reset、接收 61-D 觀察值，並確認是否能載入 10.5 秒冠軍權重。

In [ ]:
import envs
import gymnasium as gym
from stable_baselines3 import PPO

env = gym.make("Microduck-v0")
obs, info = env.reset(seed=89)
print("🦆 Microduck-v0 環境初始化成功！")
print(f"觀察空間維度: {obs.shape} (61 維度)")
print(f"動作空間維度: {env.action_space.shape} (14 個伺服舵機)")

# 檢查預訓練權重
best_ckpt = f"{GDRIVE_DIR}/checkpoints/best/best_model.zip"
if os.path.exists(best_ckpt):
    test_model = PPO.load(best_ckpt, device="cpu")
    action, _ = test_model.predict(obs, deterministic=True)
    obs, rew, term, trunc, _ = env.step(action)
    print(f"🏆 成功載入冠軍模型！單步預測動作: {action.shape}, 獲得獎勵: {rew:.2f}")
else:
    print("🌱 尚未找到預訓練模型，將從隨機權重開始訓練。")

## 步驟 5：啟動 8 隻小鴨平行強化學習訓練 (8 Clones Vectorized PPO)

在 Colab 雲端實例上啟動 8 個平行環境進行 PPO 策略優化。
訓練過程產生的最新模型會**直接寫入您的 Google Drive**，永久不丟失！

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from envs.microduck_env import MicroduckEnv

NUM_CLONES = 8
TRAIN_TIMESTEPS = 300_000

def make_env_fn(clone_id):
    def _thunk():
        env = MicroduckEnv(
            render_mode="rgb_array",
            render_width=320,
            render_height=240,
        )
        return Monitor(env)
    return _thunk

print(f"🦆 正在建立 {NUM_CLONES} 個平行小鴨環境...")
vec_env = DummyVecEnv([make_env_fn(i) for i in range(NUM_CLONES)])

policy_kwargs = dict(
    net_arch=dict(pi=[256, 128, 64], vf=[256, 128, 64]),
)

# 接續訓練或全新訓練
resume_ckpt = f"{GDRIVE_DIR}/checkpoints/best/best_model.zip"
if os.path.exists(resume_ckpt):
    print(f"📂 載入已儲存之最佳權重接續訓練：{resume_ckpt}")
    model = PPO.load(
        resume_ckpt,
        env=vec_env,
        device="auto",
        tensorboard_log=f"{GDRIVE_DIR}/logs",
        learning_rate=1e-4,
        gamma=0.9985,
    )
else:
    print("🌱 從隨機權重初始化 PPO 神經網絡...")
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=64,
        n_epochs=10,
        gamma=0.9985,
        policy_kwargs=policy_kwargs,
        verbose=1,
        device="auto",
        tensorboard_log=f"{GDRIVE_DIR}/logs",
    )

print(f"🚀 開始執行 {TRAIN_TIMESTEPS:,} 步平行訓練...")
model.learn(total_timesteps=TRAIN_TIMESTEPS)

# 將模型永久儲存至 Google Drive
saved_model_path = f"{GDRIVE_DIR}/checkpoints/microduck_colab_trained.zip"
model.save(saved_model_path)
print(f"\n✅ 訓練完成！模型已永久保存至 Google Drive：{saved_model_path}")

## 步驟 6：評估步態、錄製高畫質影片並於 Colab 中即時播放

載入最新訓練出來的模型，進行 50 Hz 步態模擬，並渲染帶有抬頭顯示器 (HUD: 時間、速度、高度、雙腳著地) 的 MP4 影片，直接在瀏覽器儲存格內播放！

In [ ]:
import cv2
import numpy as np
from IPython.display import HTML
from base64 import b64encode
from envs.microduck_env import MicroduckEnv
from stable_baselines3 import PPO

print("🎥 正在評估步態並渲染高畫質 HUD 影片...")
eval_model_path = f"{GDRIVE_DIR}/checkpoints/microduck_colab_trained.zip"
if not os.path.exists(eval_model_path):
    eval_model_path = f"{GDRIVE_DIR}/checkpoints/best/best_model.zip"

eval_model = PPO.load(eval_model_path, device="cpu")
eval_env = MicroduckEnv(
    render_mode="rgb_array",
    render_width=640,
    render_height=480,
    max_episode_steps=350,
)
eval_env.resample_commands = False

obs, _ = eval_env.reset(seed=89)
eval_env.commands[0] = 0.08
obs[48:51] = [0.08, 0.0, 0.0]

frames = []
for s in range(350):
    rgb = eval_env.render()
    if rgb is not None:
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        z = float(eval_env.data.qpos[2])
        site_mat = eval_env.data.site_xmat[eval_env.imu_site_id].reshape(3, 3)
        linvel = site_mat.T @ eval_env.data.qvel[:3]
        t_s = s * 0.02

        # 著地偵測
        lc, rc = False, False
        for c_idx in range(eval_env.data.ncon):
            c = eval_env.data.contact[c_idx]
            g1, g2 = c.geom1, c.geom2
            if (g1 == eval_env.floor_geom_id and g2 == eval_env.left_foot_geom_id) or (g2 == eval_env.floor_geom_id and g1 == eval_env.left_foot_geom_id):
                lc = True
            if (g1 == eval_env.floor_geom_id and g2 == eval_env.right_foot_geom_id) or (g2 == eval_env.floor_geom_id and g1 == eval_env.right_foot_geom_id):
                rc = True

        # 繪製 HUD
        overlay = bgr.copy()
        cv2.rectangle(overlay, (0, 0), (640, 75), (15, 20, 28), -1)
        cv2.addWeighted(overlay, 0.75, bgr, 0.25, 0, bgr)
        cv2.putText(bgr, f"Time: {t_s:.2f}s | Step: {s} | Height: {z:.3f}m | Vel: {linvel[0]:+.2f}m/s", (16, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (46, 196, 182), 2)
        cv2.putText(bgr, f"Stance: Left [{ 'CONTACT' if lc else 'SWING' }] | Right [{ 'CONTACT' if rc else 'SWING' }]", (16, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (220, 220, 220), 1)
        frames.append(bgr)

    action, _ = eval_model.predict(obs, deterministic=True)
    obs, rew, term, trunc, _ = eval_env.step(action)
    obs[48:51] = [0.08, 0.0, 0.0]
    if term or trunc:
        print(f"行走在第 {s} 步完成 (持續 {t_s:.2f} 秒)！")
        break

# 儲存為 MP4 檔案至 Google Drive
video_file = f"{GDRIVE_DIR}/videos/microduck_colab_walk.mp4"
vw = cv2.VideoWriter(video_file, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (640, 480))
for f in frames:
    vw.write(f)
vw.release()
print(f"✅ 影片已儲存至 Google Drive：{video_file}")

# 轉碼為 H.264 以便在瀏覽器中播放
web_video = "/tmp/colab_preview.mp4"
!ffmpeg -y -i {video_file} -vcodec libx264 -crf 23 {web_video} -loglevel quiet

mp4_data = open(web_video, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()
HTML(f'''
<div style="text-align:center;">
  <h3 style="color:#2ec4b6;">🦆 Microduck 雙足行走即時評估影片 (50 Hz 步態)</h3>
  <video width="640" height="480" controls autoplay loop style="border-radius:10px; border:2px solid #2ec4b6;">
    <source src="{data_url}" type="video/mp4">
  </video>
</div>
''')

## 步驟 7：匯出實機 ONNX 模型 (Sim2Real)

將訓練好的 PyTorch / SB3 策略神經網絡匯出為符 Pollen Robotics `robotd` 規範的 ONNX 格式，可直接部署至實體機器人晶片。

In [ ]:
import torch
from stable_baselines3 import PPO

onnx_output_path = f"{GDRIVE_DIR}/checkpoints/microduck_policy.onnx"
print(f"📦 正在將權重匯出至 ONNX 格式：{onnx_output_path}")

model = PPO.load(eval_model_path, device="cpu")
actor = model.policy.mlp_extractor.policy_net
action_net = model.policy.action_net

class PolicyWrapper(torch.nn.Module):
    def __init__(self, actor, action_net):
        super().__init__()
        self.actor = actor
        self.action_net = action_net
    def forward(self, obs):
        features = self.actor(obs)
        return self.action_net(features)

wrapper = PolicyWrapper(actor, action_net)
wrapper.eval()

dummy_obs = torch.zeros((1, 61), dtype=torch.float32)
torch.onnx.export(
    wrapper,
    dummy_obs,
    onnx_output_path,
    input_names=["obs"],
    output_names=["actions"],
    dynamic_axes={"obs": {0: "batch_size"}, "actions": {0: "batch_size"}},
    opset_version=14,
)

print(f"✅ ONNX 實機模型已匯出並保存於 Google Drive：{onnx_output_path}")
!ls -lh {onnx_output_path}